In [ ]:
import numpy as np
import zipfile
import pandas as pd
import glob
import json
import seaborn as sns
import spacy
import nltk
from IPython.core.display import HTML
import matplotlib.pyplot as plt
import scipy

In [ ]:
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_core_sci_md-0.5.3.tar.gz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.1/119.1 MB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for en-core-sci-md: filename=en_core_sci_md-0.5.3-py3-none-any.whl size=119160713 sha256=512f99a350404a7752bde13189ae96927bd247325b29ae1a56f7f43dbc8666e0
  Stored in directory: /root/.cache/pip/wheels/17/6d/93/c68ae1956d1f1ea08abb18e2a6f4e21a8b6aa84f607d495224
Successfully built en-core-sci-md


In [ ]:
import en_core_sci_md

In [ ]:
path = '/content/drive/MyDrive/CORD-19-research-challenge.zip'
zip_object = zipfile.ZipFile(file=path, mode ='r')
zip_object.extractall('./')
zip_object.close()

In [ ]:
corona_features = {'paper_id': [], 'title': [],
                   'abstract': [], 'text': []}

In [ ]:
type(corona_features)

dict

In [ ]:
corona_df = pd.DataFrame.from_dict(corona_features)

In [ ]:
type(corona_df)

pandas.core.frame.DataFrame

In [ ]:
json_filenames = glob.glob(f'{"./"}//**/*.json', recursive = True)

In [ ]:
print(json_filenames)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
len(json_filenames)

29317

In [ ]:
def return_corona_df(json_filenames, df=None):
    if df is None:
        df = pd.DataFrame(columns=['paper_id', 'title', 'abstract', 'text'])

    for arquivo in json_filenames:
        if arquivo == './sample_data/anscombe.json':
            continue

        try:
            with open(arquivo) as json_data:
                data = json.load(json_data)

                paper_id = data.get('paper_id', '').strip()
                title = data['metadata'].get('title', '').strip()
                abstract = '\n'.join([abstract['text'] for abstract in data.get('abstract', [])]).strip()
                text = '\n'.join([text['text'] for text in data.get('body_text', [])]).strip()

                row = {'paper_id': paper_id, 'title': title, 'abstract': abstract, 'text': text}

                df = df.append(row, ignore_index=True)
        except Exception as e:
            print(f"Error processing {arquivo}: {e}")

    return df

In [ ]:
corona_df = return_corona_df(json_filenames, corona_df)

In [ ]:
corona_df.head()

,paper_id,title,abstract,text
0,5779e410753e13a0a79f9f872321fe42df49e65c,Structural Insight Into the Viral 3C-Like Prot...,"publicly funded repositories, such as the WHO ...","In early 2003, about 8500 people were diagnose..."
1,dcd26fec2c7b83e1cbf244ce75aed864af4d4489,Functional polymorphisms of the CCL2 and MBL g...,"Journal of Infection (2015) 71, 101e109 Severe...",Functional polymorphisms of the CCL2 and MBL g...
2,829290abbfe9846568279740999b78c35e75b224,,,"Dannevig, B.H., see Grove, S. (132) 104 Das, D..."
3,c0399922a636e04aa6b5ffe2989e4d147ccd5626,Lactobacillus acidophilus S-layer protein-medi...,To gain insight into the mechanism of Lactobac...,Surface layer (S-layer) proteins are crystalli...
4,2144267cf424681c6f75208bd17198b59e630fe2,Human Challenge Pilot Study with Cyclospora ca...,We describe a pilot study that attempted to in...,highest oocyst counts were purified and concen...


In [ ]:
corona_df.shape

(29315, 4)

In [ ]:
sns.heatmap(corona_df.isnull());

In [ ]:
corona_df.duplicated().sum()

0

In [ ]:
def Empty_Strings(corona_df):
  for i in corona_df:
    vazio = len(corona_df[corona_df[i] == ""])
    print(f'Coluna: {i}, Strings Vazias: {vazio}')

In [ ]:
Empty_Strings(corona_df)

Coluna: paper_id, Strings Vazias: 0
Coluna: title, Strings Vazias: 3583
Coluna: abstract, Strings Vazias: 8051
Coluna: text, Strings Vazias: 0


In [ ]:
for i in corona_df:
  corona_df = corona_df[corona_df[i] != ""]

In [ ]:
Empty_Strings(corona_df)

Coluna: paper_id, Strings Vazias: 0
Coluna: title, Strings Vazias: 0
Coluna: abstract, Strings Vazias: 0
Coluna: text, Strings Vazias: 0


In [ ]:
corona_df.shape

(20343, 4)

In [ ]:
corona_df.duplicated().sum()

0

In [ ]:
corona_df.drop_duplicates(['abstract', 'text', 'title'], inplace=True)

In [ ]:
corona_df.shape

(20211, 4)

In [ ]:
data = corona_df.sample(n=500, random_state=1)

In [ ]:
data.shape

(500, 4)

In [ ]:
data.head()

,paper_id,title,abstract,text
11293,c455cac4ef5d0fc056295a5b421e92b1489b693e,Dual infections of feeder pigs with porcine re...,Dual infections of pigs with porcine reproduct...,"groups 1, 2 and 3 respectively. Mean weight ga..."
13704,776a2bc2e0eb29fd2cf604d9f9a335fb9ead0528,Effect of chito-oligosaccharide on the oral ab...,"Phenolic acids, the main active ingredients in...","Flos Lonicerae, a flower of Lonicera japonica ..."
1457,c8674c3258dcbccdc06d449458b75ee44ada132d,Toona sinensis Roem tender leaf extract inhibi...,Aim of the study: Severe acute respiratory syn...,Severe acute respiratory syndrome (SARS) is a ...
11730,db211cc89d8a950a96b6bc9f04894c175b80a99a,Consecutive pathological and immunological alt...,The development of intestinal lesions after in...,Swine dysentery is a widespread enteric diseas...
1054,0d8f8313ebf345a47ae755d7f470b393a79893fe,A putative diacidic motif in the SARS-CoV ORF6...,The ORF6 protein is one of the eight accessory...,An outbreak of Severe Acute Respiratory Syndro...


In [ ]:
sample_text = data['text'][11730]
sample_text

In [ ]:
nlp = en_core_sci_md.load(disable=['tagger', 'parser', 'ner'])
nlp.max_lenght = 2000000

/usr/local/lib/python3.10/dist-packages/spacy/language.py:2141: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [ ]:
print (spacy.lang.en.stop_words.STOP_WORDS)

{'hence', 'two', 'really', 'various', 'own', 'did', 'call', 'amongst', "'re", 'otherwise', 'whether', 'his', 'latter', 'alone', 'us', 'by', 'unless', 'whereas', 'the', 'myself', 'hereupon', "n't", 'do', 'put', 'no', 'hers', 'others', 'rather', 'she', 'sometimes', 'it', 'somewhere', 'never', 'nowhere', 'without', 'has', 'so', 'back', 'top', 'must', 'become', 'our', 'elsewhere', 'keep', 'at', 'under', 'anything', 'sometime', 'indeed', 'their', 'first', 'in', 'nine', 'until', 'himself', 'enough', 'them', 'less', 'already', 'full', 'both', 'mine', 'towards', 'none', 'that', 'which', 'five', '‘d', 'sixty', 'almost', 'afterwards', 'into', '’re', 'hereafter', 'hundred', "'ve", 'each', 'and', 'now', 'few', 'there', 'me', 'beyond', 'thereafter', '’d', 'front', 'may', 'therefore', 'whom', 'beforehand', 'least', 'nor', 'take', 'another', 'does', 'thereupon', 'thus', 'him', 'ours', 're', 'before', 'since', 'make', 'around', 'any', 'themselves', '‘m', 'noone', 'once', 'whereupon', 'cannot', 'most',

In [ ]:
len(spacy.lang.en.stop_words.STOP_WORDS)

326

In [ ]:
new_stop_words = ['et', 'al', 'doi', 'copyright', 'https', 'fig', 'table', 'result', 'show']
for word in new_stop_words:
  nlp.vocab[word].is_stop= True

In [ ]:
def spacy_tokenizer(sentence):
    sentence = sentence.lower()

    tokens = [word.lemma_ for word in nlp(sentence) if not (word.is_stop or
                                                            word.like_num or
                                                            word.is_punct or
                                                            word.is_space or
                                                            len(word) == 1)]

    result = ' '.join([str(element) for element in tokens if element != ' '])

    return result

In [ ]:
result = spacy_tokenizer(sample_text)
result

In [ ]:
sample_text

In [ ]:
data['text'] = data['text'].apply(spacy_tokenizer)

In [ ]:
import os

for index, row in data.iterrows():
    os.makedirs('corpus', exist_ok=True)
    text_file = open('corpus/' + row['paper_id'] + '.txt', 'w')
    n = text_file.write(row['text'])
    text_file.close()
print("Arquivos de texto criados com sucesso no diretório 'corpus'.")

Arquivos de texto criados com sucesso no diretório 'corpus'.


In [ ]:
from nltk.corpus import PlaintextCorpusReader

corpus = PlaintextCorpusReader('corpus', '.*')
files = corpus.fileids()

In [ ]:
words = corpus.words()
words

['addition', 'preventative', 'care', 'nutritional', ...]

In [ ]:
len(words)

1389323

In [ ]:
frequency = nltk.FreqDist(words)
most_common = frequency.most_common(100)
most_common

[('-', 54472),
 ('.', 19703),
 ('/', 10112),
 ('cells', 7999),
 ('virus', 6330),
 ('al', 6159),
 ('infection', 4916),
 ('protein', 4115),
 ('cell', 3961),
 ('study', 3927),
 ('viral', 3778),
 ('data', 3621),
 ('1', 2900),
 ('health', 2887),
 ('de', 2882),
 ('fig', 2847),
 ('patients', 2826),
 ("'", 2780),
 ('viruses', 2768),
 ('rna', 2732),
 ('expression', 2719),
 ('figure', 2707),
 ('infected', 2695),
 ('human', 2653),
 ('results', 2651),
 ('time', 2592),
 ('control', 2561),
 ('studies', 2558),
 ('disease', 2555),
 ('analysis', 2432),
 ('samples', 2376),
 ('proteins', 2329),
 ('number', 2273),
 ('different', 2189),
 ('high', 2170),
 ('e', 2161),
 ('specific', 2112),
 ('group', 2110),
 ('l', 2108),
 ('ml', 2103),
 ('pcr', 2082),
 ('based', 2065),
 ('found', 2046),
 ('sars', 2032),
 ('mice', 2029),
 ('shown', 1994),
 ('reported', 1975),
 ('cov', 1961),
 ('respiratory', 1823),
 ('activity', 1799),
 ('cases', 1789),
 ('associated', 1787),
 ('dna', 1768),
 ('observed', 1748),
 ('positive',

In [ ]:
from matplotlib.colors import  ListedColormap
color_map = ListedColormap(['orange', 'green', 'red', 'magenta'])

In [ ]:
!pip install wordcloud

In [ ]:
from wordcloud import WordCloud  # Import the WordCloud class

cloud = WordCloud(background_color='white', max_words=100, colormap=color_map)

In [ ]:
cloud = cloud.generate(data['text'].str.cat(sep='\n'))
plt.figure(figsize=(15, 15))
plt.imshow(cloud)
plt.axis('off')
plt.show()

In [ ]:
data.to_csv('corona.csv')

In [ ]:
text = str(data['text'][11730])
text

In [ ]:
nlp_ent = spacy.load("en_core_web_sm")

nlp_ent.max_lenght = 2000000

In [ ]:
doc = nlp_ent(text)

In [ ]:
for entity in doc.ents:
  if entity.label_ == 'NORP' or entity.label_ == 'GPE':
    print (entity.text, entity.label_)

al. GPE
al. GPE
al. GPE
al. GPE
swedish NORP
anaesthesia GPE
organisms/ NORP
japan GPE
hematoxylin GPE
newcastle GPE
united kingdom GPE
hematoxylin GPE
italy GPE
ncfa GPE
ncfa GPE
ncfa GPE
al. GPE
germany GPE
los angeles usa GPE
al. GPE
al. GPE
al. GPE
al. GPE
al. GPE


In [ ]:
from spacy import displacy
displacy.render(doc, style='ent', jupyter= True)

In [ ]:
gpe = []
for index, row in data.iterrows():
  text = row['text']
  doc = nlp_ent(text)
  for entity in doc.ents:
    if entity.label_ == 'GPE':
      gpe.append(str(entity.text))

In [ ]:
print(gpe)

In [ ]:
values_gpe, counts_gpe = np.unique(np.array(gpe), return_counts=True)

In [ ]:
gpe_df = pd.DataFrame({'value':values_gpe, 'counts':counts_gpe})

In [ ]:
gpe_df.shape

(1200, 2)

In [ ]:
gpe_df = gpe_df[gpe_df.counts > 50]
gpe_df.shape

(17, 2)

In [ ]:
sns.set(rc={'figure.figsize':(15, 8)})
sns.barplot(x='value', y='counts', data=gpe_df);

In [ ]:
text = nltk.Text(corpus.words())

In [ ]:
matches = text.concordance('virus', width=150, lines=30)

Displaying 30 of 6330 matches:
nfections surpassed deaths historically children underrepresented ebola virus disease evd outbreaks outbreak dynamics societal structure reliable est
n previously treated time appropriate assessment likely prognosis ebola virus appeared high - income countries little doubt healthcare systems effect
erall health population doubt critical care public health overlap ebola virus outbreak mobile population carries densely populated areas adequate cri
ionally relevant phenotype interest case host species context dense rna virus genomes high degree linkage analyse polymerase gene sequences rna virus
d high level transmissibility humans pathogen rapidly identified origin virus initially traced palm civets bats identified natural reservoirs sars - 
rs patients collected early mid - stages epidemic compatible adaptation virus new host species late samples variable suggesting selective pressures s
tion shaped host phylogeny streicker colleagues defined lineages rabi

In [ ]:
type(matches)

NoneType

In [ ]:
string = 'addition preventative care nutritional support development effective pediatric emergency critical care services resource-limited countries substantially'
search_string = 'preventative'

In [ ]:
print(string.find(search_string))

9


In [ ]:
string[9:9+10]

'preventati'

In [ ]:
string[9-9:9]

'addition '

In [ ]:
def find_texts(input_str, search_str, number_of_words):
    """
    Finds occurrences of search_str in input_str, along with surrounding words.

    Args:
        input_str (str): The input string to search.
        search_str (str): The string to search for.
        number_of_words (int): The number of surrounding words to include.

    Returns:
        list: A list of strings containing the search_str and surrounding words.
    """

    results = []  # Use a descriptive variable name
    words = input_str.split()  # Split input_str into words for efficient searching

    for i in range(len(words)):
        if words[i] == search_str:
            start = max(0, i - number_of_words)  # Ensure proper boundary handling
            end = min(i + number_of_words + 1, len(words))
            results.append(" ".join(words[start:end]))

    return results

In [ ]:
text = find_texts(string, search_string, 50)
text

['addition preventative care nutritional support development effective pediatric emergency critical care services resource-limited countries substantially']

In [ ]:
display(HTML(f'<h2>{search_string.upper()}</h2>'))
display(HTML(f"""<p><strong>Number of matches:</strong>{len(text)}</p>"""))
for i in text:
  market_text = str(i.replace(search_string, f'<mark>{search_string}</mark>'))
  display(HTML(f"""<blockquote>... {market_text} ... </blockquote>"""))

In [ ]:
def find_all_texts(input_str, search_str, number_of_words):
    """
    Finds all occurrences of search_str in input_str, along with surrounding words,
    ensuring efficient word splitting and boundary handling.

    Args:
        input_str (str): The input string to search.
        search_str (str): The string to search for.
        number_of_words (int): The number of surrounding words to include.

    Returns:
        list: A list of strings containing the search_str and surrounding words.
    """

    words = input_str.split()  # Split into words for efficient search
    results = []

    for i in range(len(words)):
        if words[i] == search_str:
            start = max(0, i - number_of_words)  # Ensure proper boundary handling
            end = min(i + number_of_words + 1, len(words))
            results.append(" ".join(words[start:end]))

    return results

In [ ]:
search_string = 'preventative'
search_string = spacy_tokenizer(search_string)
search_string

'preventative'

In [ ]:
documentos = []

for index, row in data.iterrows():
    doc_list = find_texts(row['text'], search_string, 40)
    documentos.append(doc_list)

In [ ]:
len(documentos)

500

In [ ]:
for doc in documentos:
    if doc:
        print(doc)

['disease surprisingly pattern changing recent human case documented primary cerebral insult post-hev infection hev infection demonstrates high case fatality rate humans estimated currently curative treatment exists apart off-label use monoclonal antibodies best supportive anecdotal application current treatment strategies center prophylactic preventative strategies endorsed biosecurity queensland involves culling suspect and/or infected equines regardless symptom status contact isolation humans concerning potential equine hev cases and/ use personal protective equipment strategies contain spread virus targeting bats futile given wide changing geographical habitats bats use', 'development human hev vaccine time-consuming prohibitively costly current stage strategy feasible concerning containment emerging infection clearly evidenced taking account responses recent emergence bat-related zoonotic epidemics ebola mers-cov hev poses significant challenge public health affecting equines cani

In [ ]:
for index, row in corona_df.iterrows():
    texts = find_all_texts(row['text'], search_string, 400)

    if not texts:
        continue

    paper_id = row['paper_id']
    title = row['title']

    display(HTML(f'<h1>{search_string.upper()}</h1>'))
    display(HTML(f"""<p>
                      <strong>Title:</strong> {title}<br>
                      <strong>ID:</strong> {paper_id}<br>
                      <strong>Number of matches:</strong> {len(texts)}
                   </p>"""))

    for text in texts:
        marked_text = text.replace(search_string, f"<mark>{search_string}</mark>")
        display(HTML(f"""<blockquote>... {marked_text} ...</blockquote>"""))

In [ ]:
string = 'addition preventative care nutritional support development effective pediatric emergency critical care services resource-limited countries substantially'
search_string = ['preventative', 'emergency']
tokens_list = [nlp(item) for item in search_string]

/usr/local/lib/python3.10/dist-packages/spacy/pipeline/lemmatizer.py:211: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [ ]:
tokens_list

[preventative, emergency]

In [ ]:
from spacy.matcher import PhraseMatcher
matcher = PhraseMatcher(nlp.vocab)
matcher.add('SEARCH', None, *tokens_list)

In [ ]:
doc = nlp(string)
matches = matcher(doc)
matches

/usr/local/lib/python3.10/dist-packages/spacy/pipeline/lemmatizer.py:211: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


[(8661325627334373315, 1, 2), (8661325627334373315, 8, 9)]

In [ ]:
search_string = ['smoking', 'pulmonary disease']
tokens_list = [nlp(item) for item in search_string]
tokens_list

[smoking, pulmonary disease]

In [ ]:
matcher = PhraseMatcher(nlp.vocab)
matcher.add('SEARCH', None, *tokens_list)
number_of_word = 50

In [ ]:
search_string_html = ' '.join([str(element)for element in search_string])
search_string_html

'smoking pulmonary disease'

In [ ]:
for index, row in data.iterrows():
  marked_text = ''
  doc = nlp(row['text'])
  paper_id = row['paper_id']
  title = row['title']
  matches = matcher(doc)

  if not matches:
    continue

  display(HTML(f'<h1>{search_string_html.upper()}</h1>'))
  display(HTML(f"""<p>
                      <strong>Title:</strong> {title}<br>
                      <strong>ID:</strong> {paper_id}<br>
                      <strong>Number of matches:</strong> {len(matches)}
                   </p>"""))

  for i in matches:
    start = i[1] - number_of_word
    if start < 0:
      start = 0

    for j in range(len(tokens_list)):
      if doc [i[1] :i[2]].similarity(tokens_list[j]) == 1.0:
        search_text = str(tokens_list[j])
        marked_text += str(doc[start:i[2] + number_of_word]).replace(search_text, f'<mark>{search_text}</mark>')
        marked_text += '<br /><br />'

  display(HTML(f"""<blockquote> ... {marked_text} ....</blockquote>"""))